<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/VecAdd_CuTile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Install the cuda.tile library from PyPI to resolve the ModuleNotFoundError
!pip install cuda.tile
!pip install cuda-tile[tileiras]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 44.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 MB 18.3 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 MB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 MB 51.1 MB/s  0:00:01
  Attempting uninstall: cuda-toolkit
    Found existing installation: cuda-toolkit 13.0.3.0
    Uninstalling cuda-toolkit-13.0.3.0:
      Successfully uninstalled cuda-toolkit-13.0.3.0
  Attempting uninstall: nvidia-nvvm
    Found existing installation: nvidia-nvvm 13.0.88
    Uninstalling nvidia-nvvm-13.0.88:
      Successfully uninstalled nvidia-nvvm-13.0.88
  Attempting uninstall: nvidia-nvjitlink
    Found existing installation: nvidia-nvjitlink 13.4.92
    Uninstalling nvidia-nvjitlink-13.4.92:
      Successfully uninstalled nvidia-nvjitlink-13.4.92
  Attempting uninstall: nvidia-cuda-runtime
    Found existi

In [9]:
"""
========================================================================================
LESSON: HIGH-PERFORMANCE VECTOR ADDITION WITH CUDA.TILE (CUTILE)
========================================================================================

1. HIGH-LEVEL OVERVIEW & CONCEPT
--------------------------------
Vector addition is the fundamental 'Hello World' of parallel computing. Given two large vectors
X and Y of size N, we compute a third vector SUM such that:
    SUM[i] = X[i] + Y[i]  for all i in [0, N-1]

In a standard CPU sequential loop, this takes O(N) time because one CPU core processes each
element one-by-one.
With GPU acceleration, we break down the large N-element vectors into smaller chunks called
'TILES' (of size BLOCK_SIZE/TILE_SIZE). Multiple independent GPU Thread Blocks process these
tiles concurrently in parallel.

2. PARALLEL EXECUTION MODEL & THE "CUTILE" PARADIGM
----------------------------------------------------
Traditional CUDA programming requires managing raw thread indices, memory coalescing, and manual
shared memory bounds. `cuda.tile` (cutile) abstracts this by focusing on 'Tiles' as first-class objects.

- GRID: A collection of Thread Blocks. Here, Grid size = Ceil(N / BLOCK_SIZE). Each Block
  is mapped to an index called 'bid' (Block ID).
- TILE: A slice of data that fits into a GPU block processor. Instead of handling elements
  individually, we load an entire chunk (tile) of size `TILE` from global GPU memory into registers,
  perform vector addition on the entire tile at once, and store the resulting tile back to GPU memory.

3. DETAILED HARDWARE & EXECUTION FLOW DIAGRAM
-----------------------------------------------

   HOST (CPU)                                        DEVICE (GPU) GLOBAL MEMORY
+-----------------------+                         +-------------------------------------+
|  X (PyTorch Tensor)   | ----(H2D Transfer)----> | x_ptr  [===Tile 0===][===Tile 1===] |
|  Y (PyTorch Tensor)   | ----(H2D Transfer)----> | y_ptr  [===Tile 0===][===Tile 1===] |
+-----------------------+                         +-------------------------------------+
           |                                                         |
    Launches Kernel                                            Parallel Load
           v                                                         v
+-----------------------+                         +-------------------------------------+
| ct.launch(grid, ...)  |                         |    GPU MULTIPROCESSOR (SM)          |
+-----------------------+                         |                                     |
                                                  |   [Block 0 (bid=0)]                 |
                                                  |     -> ct.load(x_ptr, (0,), TILE)   |
                                                  |     -> ct.load(y_ptr, (0,), TILE)   |
                                                  |     -> sum_tile = x_tile + y_tile   |
                                                  |     -> ct.store(sum_ptr, (0,), tile)|
                                                  |                                     |
                                                  |   [Block 1 (bid=1)]                 |
                                                  |     -> ct.load(x_ptr, (1,), TILE)   |
                                                  |     -> ct.load(y_ptr, (1,), TILE)   |
                                                  |     -> sum_tile = x_tile + y_tile   |
                                                  |     -> ct.store(sum_ptr, (1,), tile)|
                                                  +-------------------------------------+
                                                                     |
                                                               Parallel Store
                                                                     v
                                                  +-------------------------------------+
                                                  | sum_ptr[===Tile 0===][===Tile 1===] |
                                                  +-------------------------------------+
"""

# Import the CUDA Tile abstraction library which simplifies writing high-performance CUDA kernels in Python
import cuda.tile as ct
# Import PyTorch to handle tensor allocations, memory management, and host-side execution
import torch
# Import math to perform ceiling division when calculating the grid dimensions
import math

# Define the execution block size (the number of elements processed per individual GPU block/tile)
BLOCK_SIZE = 1024

# Decorator indicating that the following Python function is a compiled GPU kernel managed by cuda.tile
@ct.kernel
def vec_add_kernel (
    # x_ptr: Pointer to the first input vector in GPU global memory with ct.Pointer annotation
    x_ptr: ct.IndexedWithInt64,
    # y_ptr: Pointer to the second input vector in GPU global memory with ct.Pointer annotation
    y_ptr: ct.IndexedWithInt64,
    # sum_ptr: Pointer to the output destination vector in GPU global memory with ct.Pointer annotation
    sum_ptr: ct.IndexedWithInt64,
    # TILE: A constant integer defining the size of each tile processed by a block
    TILE: ct.Constant[int]
):
  # Retrieve the Block ID (bid) along the X-axis (axis=0) to identify which chunk this block is responsible for
  bid = ct.bid(axis=0)

  # Load a tile of size TILE from x_ptr in global memory, offsetting by our block ID 'bid'
  x_tile = ct.load(x_ptr, index=(bid,), shape=(TILE,))

  # Load a tile of size TILE from y_ptr in global memory, offsetting by our block ID 'bid'
  y_tile = ct.load(y_ptr, index=(bid,), shape=(TILE,))

  # Perform element-wise parallel vector addition on the loaded registers/SRAM tiles
  sum_tile = x_tile + y_tile

  # Write the computed sum_tile back to the global memory output pointer (sum_ptr) at the block index block ID
  ct.store(sum_ptr, index=(bid,), tile=sum_tile)


# Python helper function to prepare PyTorch inputs, launch the GPU kernel, and return results
def vec_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
  # Pre-allocate the result tensor on the same GPU device and with the exact shape/datatype as input x
  sum = torch.zeros_like(x)

  # Query the total number of scalar elements inside our vector
  num_elements = sum.numel()

  # Calculate grid size (number of blocks). We use ceiling division to ensure we cover any leftover elements
  grid = (math.ceil(num_elements / BLOCK_SIZE), 1, 1)

  # Launch the GPU kernel using the current active PyTorch CUDA stream
  ct.launch(
      # Pass the current active stream to ensure execution is correctly ordered with other PyTorch operations
      torch.cuda.current_stream(),
      # Configure the execution grid geometry (number of parallel thread blocks to spawn)
      grid,
      # Specify the compiled GPU kernel function to execute
      vec_add_kernel,
      # Provide kernel arguments matching the signature: (x_ptr, y_ptr, sum_ptr, TILE)
      (x, y, sum, BLOCK_SIZE)
  )

  # Return the output PyTorch tensor containing the final computed vector addition
  return sum


def main():
  # Set manual seed on host to guarantee reproducible tensor values during execution
  torch.manual_seed(0)

  # Set the length of our testing vectors (using a multiple of BLOCK_SIZE for clean boundaries)
  N = 100000

  # Create random vector X on CUDA device with float32 values between 0 and 1
  x = torch.rand(N, device="cuda", dtype=torch.float32)

  # Create random vector Y on CUDA device with float32 values between 0 and 1
  y = torch.rand(N, device="cuda", dtype=torch.float32)

  # Execute our custom written parallel cuda.tile-based vector addition function
  custom_out = vec_add(x, y)

  # Compute reference baseline result using standard PyTorch built-in addition on GPU
  torch_out = x + y

  # Compare results between custom kernel and PyTorch reference up to a tiny numerical tolerance
  success = torch.allclose(custom_out, torch_out, atol=1e-5)

  # Print validation output confirmation and boolean verification flag indicating status
  print(f"Vector Addition Success: {success}")

  # Compute absolute maximum difference between custom kernel output and expected PyTorch output
  max_diff = torch.max(torch.abs(custom_out - torch_out)).item()

  # Print computed absolute maximum numerical difference for diagnostic precision assessment
  print(f"Max Absolute Difference: {max_diff}")


# standard entrypoint block ensuring main runs only if file is executed as primary program
if __name__ == "__main__":
  # execute the initialized main function routine
  main()


Vector Addition Success: True
Max Absolute Difference: 0.0
